# Final model

### Resources

- https://geemap.org/notebooks/46_local_rf_training/

## Setup

In [ ]:
import os
import time

import ee
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from geemap import ml
from google.colab import drive
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

In [ ]:
drive.mount("/content/drive")

In [ ]:
%cd /content/drive/MyDrive/land_cover_classification_kaza

In [ ]:
user_id = "ee-alexvmt"
asset_name = "mufunta_random_forest"

In [ ]:
ee.Authenticate()
ee.Initialize(project=user_id)

## Load train and test set

In [ ]:
train = pd.read_csv("data/train.csv")
train

In [ ]:
X_train = train.drop(["LC_Nr", "LC_Out", "Landcover"], axis=1)
X_train

In [ ]:
y_train = train["LC_Nr"]
y_train

In [ ]:
test = pd.read_csv("data/test.csv")
test

In [ ]:
X_test = test.drop(["LC_Nr", "LC_Out", "Landcover"], axis=1)
X_test

In [ ]:
y_test = test["LC_Nr"]
y_test

## Train random forest with default hyperparameters

In [ ]:
rf_default = RandomForestClassifier()

In [ ]:
rf_default.fit(X_train, y_train)

In [ ]:
y_train_pred = rf_default.predict(X_train)
y_test_pred = rf_default.predict(X_test)

## Evaluation

### Train set classification metrics

In [ ]:
print("Accuracy: {:0.4f}".format(accuracy_score(y_train, y_train_pred)))

In [ ]:
print("Precision: {:0.4f}".format(precision_score(y_train, y_train_pred, average="macro")))

In [ ]:
print("Recall: {:0.4f}".format(recall_score(y_train, y_train_pred, average="macro")))

In [ ]:
print("F1-Score: {:0.4f}".format(f1_score(y_train, y_train_pred, average="macro")))

In [ ]:
print(classification_report(y_train, y_train_pred))

In [ ]:
cm = confusion_matrix(y_train, y_train_pred)
cm

In [ ]:
sns.heatmap(cm, annot=True, fmt="d")
plt.show()

### Test set classifiction metrics

In [ ]:
print("Accuracy: {:0.4f}".format(accuracy_score(y_test, y_test_pred)))

In [ ]:
print("Precision: {:0.4f}".format(precision_score(y_test, y_test_pred, average="macro")))

In [ ]:
print("Recall: {:0.4f}".format(recall_score(y_test, y_test_pred, average="macro")))

In [ ]:
print("F1-Score: {:0.4f}".format(f1_score(y_test, y_test_pred, average="macro")))

In [ ]:
print(classification_report(y_test, y_test_pred))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
cm

In [ ]:
sns.heatmap(cm, annot=True, fmt="d")
plt.show()

## Train random forest with optimized hyperparameters

Get optimized hyperparameters from notebook called `optimization.ipynb`.

In [ ]:
rf_optimized = RandomForestClassifier(
    bootstrap=False,
    criterion="entropy",
    max_features=5,
    min_samples_split=4,
    n_estimators=512,
    n_jobs=1,
    random_state=1,
    warm_start=True,
)

In [ ]:
rf_optimized.fit(X_train, y_train)

In [ ]:
y_train_pred = rf_optimized.predict(X_train)
y_test_pred = rf_optimized.predict(X_test)

## Evaluation

### Train set classification metrics

In [ ]:
print("Accuracy: {:0.4f}".format(accuracy_score(y_train, y_train_pred)))

In [ ]:
print("Precision: {:0.4f}".format(precision_score(y_train, y_train_pred, average="macro")))

In [ ]:
print("Recall: {:0.4f}".format(recall_score(y_train, y_train_pred, average="macro")))

In [ ]:
print("F1-Score: {:0.4f}".format(f1_score(y_train, y_train_pred, average="macro")))

In [ ]:
print(classification_report(y_train, y_train_pred))

In [ ]:
cm = confusion_matrix(y_train, y_train_pred)
cm

In [ ]:
sns.heatmap(cm, annot=True, fmt="d")
plt.show()

### Test set classifiction metrics

In [ ]:
print("Accuracy: {:0.4f}".format(accuracy_score(y_test, y_test_pred)))

In [ ]:
print("Precision: {:0.4f}".format(precision_score(y_test, y_test_pred, average="macro")))

In [ ]:
print("Recall: {:0.4f}".format(recall_score(y_test, y_test_pred, average="macro")))

In [ ]:
print("F1-Score: {:0.4f}".format(f1_score(y_test, y_test_pred, average="macro")))

In [ ]:
print(classification_report(y_test, y_test_pred))

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
cm

In [ ]:
sns.heatmap(cm, annot=True, fmt="d")
plt.show()

## Feature importance

In [ ]:
def get_feature_importances(column_names, feature_importances):
    importances = pd.DataFrame(
        list(zip(column_names, feature_importances, strict=False)),
        columns=["Feature", "Importance"],
    )
    importances = importances.sort_values("Importance", ascending=False)

    return importances

In [ ]:
def plot_feature_importances(feature_importances, model_name):
    plt.figure(figsize=(10, 6))

    plt.bar(feature_importances["Feature"], feature_importances["Importance"])
    plt.xticks(rotation=90)

    plt.title(f"Feature Importances - {model_name}")

    plt.tight_layout()
    plt.show()

### RF default

In [ ]:
rf_default_importances = get_feature_importances(X_train.columns, rf_default.feature_importances_)
rf_default_importances.head(10)

In [ ]:
plot_feature_importances(rf_default_importances, "RF default")

### RF optimized

In [ ]:
rf_optimized_importances = get_feature_importances(
    X_train.columns, rf_optimized.feature_importances_
)
rf_optimized_importances.head(10)

In [ ]:
plot_feature_importances(rf_optimized_importances, "RF optimized")

### Compare feature importances

In [ ]:
rf_default_importances = rf_default_importances.rename(
    columns={"Importance": "Importance - RF default"}
)
rf_optimized_importances = rf_optimized_importances.rename(
    columns={"Importance": "Importance - RF optimized"}
)
importances = pd.merge(rf_default_importances, rf_optimized_importances, how="inner", on="Feature")
importances.head(10)

In [ ]:
importances.plot(
    x="Feature",
    y=["Importance - RF default", "Importance - RF optimized"],
    kind="bar",
    rot=90,
    title="Comparison of Feature Importances",
    figsize=(12, 8),
)
plt.show()

## Permutation importance

In [ ]:
def get_permutation_importances(model, X_train, y_train, X_test, y_test):
    # set parameters
    scoring = "f1_macro"
    n_repeats = 5
    n_jobs = -1
    seed = 42
    max_samples = 0.3

    print("Getting permutation importances for train set...")
    train_results = permutation_importance(
        rf_default,
        X_train,
        y_train,
        scoring=scoring,
        n_repeats=n_repeats,
        n_jobs=n_jobs,
        random_state=seed,
    )

    print("Getting permutation importances for test set...")
    test_results = permutation_importance(
        rf_default,
        X_test,
        y_test,
        scoring=scoring,
        n_repeats=n_repeats,
        n_jobs=n_jobs,
        random_state=seed,
        max_samples=max_samples,
    )

    sorted_importances_idx = train_results.importances_mean.argsort()
    feature_names = X_train.columns
    train_importances = pd.DataFrame(
        train_results.importances[sorted_importances_idx].T,
        columns=feature_names[sorted_importances_idx],
    )
    test_importances = pd.DataFrame(
        test_results.importances[sorted_importances_idx].T,
        columns=feature_names[sorted_importances_idx],
    )

    return train_importances, test_importances

In [ ]:
def plot_permutation_importances(train_importances, test_importances):
    for name, importances in zip(
        ["train", "test"], [train_importances, test_importances], strict=False
    ):
        ax = importances.plot.box(vert=False, whis=10, figsize=(8, 16))
        ax.set_title(f"Permutation importances ({name} set)")
        ax.set_xlabel("Decrease in f1_macro")
        ax.axvline(x=0, color="k", linestyle="--")
        ax.figure.tight_layout()

### RF default

In [ ]:
%%time
rf_default_train_importances, rf_default_test_importances = get_permutation_importances(
    rf_default, X_train, y_train, X_test, y_test
)
plot_permutation_importances(rf_default_train_importances, rf_default_test_importances)

### RF optimized

In [ ]:
%%time
rf_optimized_train_importances, rf_optimized_test_importances = get_permutation_importances(
    rf_optimized, X_train, y_train, X_test, y_test
)
plot_permutation_importances(rf_optimized_train_importances, rf_optimized_test_importances)

## Convert sklearn classifier object to a list of strings

In [ ]:
# convert the estimator into a list of strings
# this function also works with the ensemble.ExtraTrees estimator
start_time = time.perf_counter()
trees = ml.rf_to_strings(rf_default, X_train.columns)
end_time = time.perf_counter()
run_time = round((end_time - start_time) / 60, 2)
print("Run time: {} minutes.".format(run_time))

In [ ]:
# print the first tree to see the result
print(trees[0])

In [ ]:
print(trees[1])

In [ ]:
# number of trees we converted should equal the number of trees we defined for the model
n_trees = 100
len(trees) == n_trees

## Convert sklearn classifier to GEE classifier

At this point you can take the list of strings and save them locally to avoid training again. However, we want to use the model with EE so we need to create an ee.Classifier and persist the data on ee for best results.

In [ ]:
# create a ee classifier to use with ee objects from the trees
ee_classifier = ml.strings_to_classifier(trees)

In [ ]:
ee_classifier.getInfo()

## Save trees to the cloud

Now we have the strings in a format that ee can use, we want to save it for later use. There is a function to export a list of tree strings to a feature collection.

In [ ]:
# specify asset id where to save trees
asset_id = "projects/" + user_id + "/assets/" + asset_name + "_trees"
asset_id

In [ ]:
# kick off an export process so it will be saved to the ee asset
ml.export_trees_to_fc(trees, asset_id)

# this will kick off an export task, so wait a few minutes before moving on
# check progress here: https://code.earthengine.google.com/tasks

In [ ]:
# save ee classifier to be used in ee directly
classifier_asset_id = "projects/" + user_id + "/assets/" + asset_name + "_classifier"
task = ee.batch.Export.classifier.toAsset(ee_classifier, "saved classifier", classifier_asset_id)
task.start()

## Save trees locally

In [ ]:
out_csv = os.path.expanduser("models/trees.csv")
ml.trees_to_csv(trees, out_csv)